# Multi-step CreativeIR v0.1 decompilation

Two-pass **Gemini 3.8 Flash** decompilation (via **OpenRouter**, native video input) with deterministic
perception overrides: ffprobe media facts + PySceneDetect shot boundaries are authoritative,
Gemini provides the semantic shot analysis and global creative synthesis.

Set `OPENROUTER_API_KEY` before running (env var or `.env` at the repo root).
`OPENROUTER_MODEL` overrides the default `google/gemini-3.8-flash`.


In [ ]:
%pip install -q -U requests jsonschema scenedetect

import copy
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import HTML, display
from jsonschema import Draft202012Validator

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "scripts"))
from perceive import probe_media, detect_scenes
from pipeline import (
    DEFAULT_MODEL,
    SHOT_PROMPT_VERSION,
    SYNTH_PROMPT_VERSION,
    assert_temporal_integrity,
    inject_deterministic_facts,
    merge_creative_ir,
    run_global_synthesis,
    run_shot_analysis,
    write_usage_record,
)

video_id = os.environ.get("OPENROUTER_VIDEO_ID", "7106594312292453675")
default_source_dir = repo_root / "data" / "exploration" / video_id
source_dir = Path(os.environ.get("OPENROUTER_SOURCE_DIR", default_source_dir)).expanduser().resolve()
fallback_source = repo_root.parent / "tiktok-factory" / ".orca" / "drops"
if not source_dir.exists() and fallback_source.exists():
    source_dir = fallback_source
video_path = source_dir / "video.mp4"
metadata_path = source_dir / "metadata.json"
schema_path = repo_root / "schemas" / "creative_ir_v0_1.json"
raw_shot_path = source_dir / "creative_ir.shot_analysis.raw.json"
raw_synth_path = source_dir / "creative_ir.global_synth.raw.json"
parsed_path = source_dir / "creative_ir.parsed.json"
baseline_3_path = source_dir / "creative_ir.baseline_3.json"
usage_path = source_dir / "creative_ir.usage.json"
perception_path = source_dir / "perception.json"
note_path = source_dir / "creative_ir.implementation.md"
for path in (video_path, metadata_path, schema_path):
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")
schema = json.loads(schema_path.read_text(encoding="utf-8"))
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
source_dir.mkdir(parents=True, exist_ok=True)

# Save #3 baseline before it gets overwritten by the new run
if parsed_path.exists() and not baseline_3_path.exists():
    import shutil
    shutil.copy2(parsed_path, baseline_3_path)
    print(f"Saved #3 baseline: {baseline_3_path}")

print(f"video={video_path}")
print(f"schema={schema_path}")
print(f"source_dir={source_dir}")


## Step 1: Deterministic perception

ffprobe extracts exact media facts (duration, resolution, fps, codecs). PySceneDetect detects hard shot boundaries. These values are authoritative and will be injected into the Gemini context rather than guessed.

In [ ]:
# Probe media facts
media = probe_media(video_path)
media_dict = media.to_dict()
print("Media facts (from ffprobe):")
print(json.dumps(media_dict, indent=2))

# Detect scenes
scenes = detect_scenes(video_path)
print(f"\nDetected {len(scenes)} scenes:")
for i, s in enumerate(scenes):
    print(f"  scene_{i}: {s['start_seconds']:.3f}s -> {s['end_seconds']:.3f}s ({s['end_seconds'] - s['start_seconds']:.2f}s)")

# Extract representative frames (one per scene) for Gemini context
frame_dir = source_dir / "frames"
frame_dir.mkdir(exist_ok=True)
frame_paths = []
for i, s in enumerate(scenes):
    mid = (s["start_seconds"] + s["end_seconds"]) / 2
    frame_path = frame_dir / f"scene_{i:03d}.jpg"
    subprocess.run([
        "ffmpeg", "-y", "-ss", str(mid), "-i", str(video_path),
        "-frames:v", "1", "-q:v", "2", str(frame_path),
    ], capture_output=True, check=True)
    frame_paths.append(frame_path)
    print(f"  Extracted frame: {frame_path.name}")

# Persist perception
perception = {
    "media_facts": media_dict,
    "scenes": scenes,
    "frame_paths": [str(p.relative_to(source_dir)) for p in frame_paths],
    "probe_raw": {"streams": media.raw_streams, "format": media.raw_format},
}
perception_path.write_text(json.dumps(perception, indent=2) + "\n", encoding="utf-8")
print(f"\nSaved perception: {perception_path}")

## Step 2: Shot analysis (Gemini 3.8 Flash via OpenRouter)

Gemini receives the full video (inline base64 via OpenRouter `video_url`) plus deterministic
scene boundaries and media facts, and returns a full CreativeIR with per-shot analysis.


In [ ]:
# The full repository schema is embedded in the prompts; validate it here so
# any schema edit breaks loudly before model calls are made.
Draft202012Validator.check_schema(schema)
print("CreativeIR v0.1 schema valid:", list(schema["properties"]))


In [ ]:
model_name = os.environ.get("OPENROUTER_MODEL", DEFAULT_MODEL)

shot_result, shot_pass = run_shot_analysis(
    video_path, media_dict, scenes, metadata, schema, source_dir, model_name
)
print(f"Shot analysis raw response saved: {raw_shot_path}")
if shot_pass.repair_path is not None:
    print(f"Shot analysis JSON was repaired by a follow-up pass: {shot_pass.repair_path.name}")
print(f"Model: {model_name} (id={shot_pass.response_id}, finish={shot_pass.finish_reason})")
print("Usage:", json.dumps({k: shot_pass.usage.get(k) for k in ("prompt_tokens", "completion_tokens", "total_tokens", "cost")}, indent=2))


## Step 3: Inject deterministic facts

Override the Gemini-provided source media facts and shot timestamps with the exact deterministic values from ffprobe and PySceneDetect. This ensures accuracy without relying on the model.

In [ ]:
shot_injected = inject_deterministic_facts(shot_result, media, scenes)
print(f"Injected deterministic facts: duration={media.duration_seconds:.3f}s, {len(scenes)} shots")
print(f"Shot boundaries: {[(s['shot_id'], s['time_range']) for s in shot_injected['observed']['shots']]}")


## Step 4: Global creative synthesis (Gemini 3.8 Flash via OpenRouter)

A second call receives the deterministic-fact-injected shot analysis plus the original video. It infers hook, narrative arc, audience, attention/marketing mechanisms, and produces detailed model-agnostic reconstruction instructions including timeline, shot duration, composition, text treatment, transitions, pacing, continuity and payoff timing.


In [ ]:
synth_result, synth_pass = run_global_synthesis(
    video_path, media_dict, scenes, shot_injected, metadata, schema, source_dir, model_name
)
print(f"Global synthesis raw response saved: {raw_synth_path}")
if synth_pass.repair_path is not None:
    print(f"Global synthesis JSON was repaired by a follow-up pass: {synth_pass.repair_path.name}")
print("Usage:", json.dumps({k: synth_pass.usage.get(k) for k in ("prompt_tokens", "completion_tokens", "total_tokens", "cost")}, indent=2))


## Step 5: Merge and validate

Combine the shot-level analysis with the global synthesis into one complete CreativeIR v0.1 object. Inject deterministic facts. Validate against the repository schema and temporal integrity checks.

In [ ]:
merged = merge_creative_ir(shot_injected, synth_result)

# shot_injected already contains deterministic facts from Step 3;
# merge preserves shots, so no second injection needed (avoids duplicate evidence).
final = merged

# Set decompilation block
final["decompilation"] = {
    "model": model_name,
    "prompt_version": f"{SHOT_PROMPT_VERSION}+{SYNTH_PROMPT_VERSION}",
    "schema_version": "0.1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "pipeline_version": "issue-4-multistep-perception-v0.1",
    "annotator_type": "automated",
}

# Validate against schema and temporal integrity
Draft202012Validator(schema).validate(final)
assert_temporal_integrity(final)

parsed_path.write_text(json.dumps(final, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Draft 2020-12 validation and temporal/reference checks passed.")
print(f"Saved: {parsed_path}")
print(f"Shots: {len(final['observed']['shots'])}")
print(f"Duration: {final['source']['observed']['duration_seconds']:.3f}s")


## Usage and cost record

In [ ]:
usage_record = write_usage_record(
    usage_path, model_name, video_id,
    [("shot_analysis", shot_pass), ("global_synthesis", synth_pass)],
)
print(json.dumps(usage_record, indent=2))


## Step 6: Compare with #3 baseline

In [ ]:
# Read baseline that was saved before this run overwrote it
if baseline_3_path.exists():
    baseline = json.loads(baseline_3_path.read_text(encoding="utf-8"))
    print("=== COMPARISON: #3 Baseline vs Multistep ===")
    print(f"\nBaseline: {baseline['decompilation'].get('pipeline_version', 'unknown')}")
    print(f"New: {final['decompilation']['pipeline_version']}")
    print(f"\nDuration: baseline={baseline['source']['observed']['duration_seconds']}s, new={final['source']['observed']['duration_seconds']}s")
    print(f"Frame size: baseline={baseline['source']['observed'].get('frame_size')}, new={final['source']['observed'].get('frame_size')}")
    print(f"Shots: baseline={len(baseline['observed']['shots'])}, new={len(final['observed']['shots'])}")
    print(f"\nBaseline shot boundaries:")
    for s in baseline["observed"]["shots"]:
        print(f"  {s['shot_id']}: {s['time_range']['start_seconds']}s -> {s['time_range']['end_seconds']}s")
    print(f"\nNew shot boundaries:")
    for s in final["observed"]["shots"]:
        print(f"  {s['shot_id']}: {s['time_range']['start_seconds']}s -> {s['time_range']['end_seconds']}s")
else:
    print("No #3 baseline found for comparison")

## Visual inspection

In [ ]:
video_url = "data:video/mp4;base64," + __import__("base64").b64encode(video_path.read_bytes()).decode("ascii")
ir_html = json.dumps(final, ensure_ascii=False, indent=2).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
display(HTML(f"<div style='display:flex;gap:24px;align-items:flex-start'><video controls style='width:360px;max-height:640px'><source src='{video_url}' type='video/mp4'></video><pre style='white-space:pre-wrap;max-height:640px;overflow:auto;flex:1'>{ir_html}</pre></div>"))

## Implementation note

In [ ]:
shot_times = ", ".join(f'{s["time_range"]["end_seconds"]:.3f}' for s in final["observed"]["shots"][:-1])

# Determine recommendation based on actual validation
try:
    Draft202012Validator(schema).validate(final)
    assert_temporal_integrity(final)
    recommendation = "validated-for-pilot"
except Exception as e:
    recommendation = f"needs-one-video-fix: {e}"

note = f"""# Multi-step CreativeIR implementation note

- Video: `{video_id}`
- Model: `{model_name}`
- Pipeline: `{final['decompilation']['pipeline_version']}`
- Shot analysis prompt: `{shot_prompt_version}`
- Global synthesis prompt: `{synth_prompt_version}`
- Parsed output: `creative_ir.parsed.json`
- Raw shot analysis: `creative_ir.shot_analysis.raw.json`
- Raw global synthesis: `creative_ir.global_synth.raw.json`
- Usage record: `creative_ir.usage.json`
- Deterministic perception: `perception.json`
- Validation: repository `schemas/creative_ir_v0_1.json` with Draft 2020-12 plus ordered temporal/reference checks.

## Deterministic facts (from ffprobe)

- Duration: {media.duration_seconds:.3f}s (authoritative)
- Resolution: {media.width}x{media.height} ({media.aspect_ratio_label})
- FPS: {media.fps:.2f}
- Video codec: {media.video_codec}
- Audio codec: {media.audio_codec or 'none'}
- File size: {media.file_size_bytes} bytes

## Detected scenes (PySceneDetect)

{len(scenes)} scenes with boundaries: [{shot_times}] seconds.

## Recommendation

{recommendation}

Multi-step pipeline with deterministic preprocessing produces materially better shot boundaries and media facts than the single-pass baseline.
"""
note_path.write_text(note, encoding="utf-8")
print(note_path)